# RAG Pipeline from Scratch

**Retrieval-Augmented Generation** — Building a complete RAG system end-to-end.

> *"The most important thing about RAG is that it grounds LLM outputs in retrievable facts, reducing hallucinations and enabling verifiable answers."*

---

## What is RAG?

Retrieval-Augmented Generation (RAG) is a framework that combines **information retrieval** with **text generation**. Instead of relying solely on the knowledge baked into a language model during training, RAG:

1. **Retrieves** relevant documents from a knowledge base given a query
2. **Augments** the prompt with those retrieved documents
3. **Generates** an answer grounded in the retrieved context

**Key benefits:**
- ✅ Reduces hallucinations by grounding outputs in external knowledge
- ✅ Knowledge base can be updated without retraining the LLM
- ✅ Enables answering questions about private/custom documents
- ✅ Provides provenance — you can cite which document was used

---

## Pipeline Overview

```
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│  Corpus  │ →  │ Chunking │ →  │ Embed &  │ →  │ Retrieve │ →  │ Generate │
│ (docs)   │    │          │    │  Index   │    │          │    │          │
└──────────┘    └──────────┘    └──────────┘    └──────────┘    └──────────┘
                                                      ↓
                                               ┌──────────┐
                                               │ Rerank   │
                                               └──────────┘
```

**What we'll build in this notebook:**
| Step | Component | What it does |
|:----:|:----------|:-------------|
| 1 | **Document Corpus** | Generate a synthetic collection of data science articles |
| 2 | **Chunking** | Split documents into retrievable pieces |
| 3 | **Embeddings** | Convert chunks into dense vector representations |
| 4 | **Vector Store** | Index embeddings for fast similarity search |
| 5 | **Retrieval** | Find the most relevant chunks for a query |
| 6 | **Reranking** | Re-order retrieved chunks for better precision |
| 7 | **Generation** | Feed context + query to an LLM and get an answer |
| 8 | **Evaluation** | Measure retrieval quality and answer correctness |

---

### Learning Objectives
- Understand the RAG architecture and each component's role
- Compare chunking strategies and their trade-offs
- Build a vector store index from scratch
- Implement dense retrieval with cosine similarity
- Use cross-encoder models for reranking
- Evaluate RAG pipeline with retrieval metrics
- See how generation quality depends on retrieval quality

In [1]:
# ===== 1. Problem Definition =====
# We're building a system that can answer questions about data science concepts
# using a curated knowledge base of articles.

print("✅ Problem defined: Build a RAG system for data science Q&A")
print(f"   {'Task':12s}: Answer natural language questions about ML/DL/NLP topics")
print(f"   {'Approach':12s}: Retrieval-Augmented Generation")
print(f"   {'Knowledge':12s}: Synthetic corpus of ~60 articles across 6 domains")
print(f"   {'Success':12s}: High retrieval precision + grounded, factual answers")

✅ Problem defined: Build a RAG system for data science Q&A
   Task        : Answer natural language questions about ML/DL/NLP topics
   Approach    : Retrieval-Augmented Generation
   Knowledge   : Synthetic corpus of ~60 articles across 6 domains
   Success     : High retrieval precision + grounded, factual answers


## 2. Environment Setup

Install required packages and import all dependencies.

In [2]:
# ===== 2. Environment Setup =====
# Installing all required packages for the RAG pipeline.

import sys
!{sys.executable} -m pip install -q \
    pandas numpy matplotlib seaborn \
    sentence-transformers chromadb \
    tqdm scikit-learn \
    transformers torch --quiet 2>/dev/null

print("✅ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 39.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 27.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.3 MB/s eta 0:00:00
✅ Dependencies installed


In [26]:
# ===== Imports =====

import json
import os
import re
import warnings
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(BASE_DIR, 'data')
VIZ_DIR = os.path.join(BASE_DIR, 'visualization')
os.makedirs(VIZ_DIR, exist_ok=True)

print(f"📂 Data dir: {DATA_DIR}")
print(f"📂 VIZ dir: {VIZ_DIR}")
print("✅ Imports complete")

📂 Data dir: /data
📂 VIZ dir: /visualization
✅ Imports complete


In [36]:
cwd = os.getcwd()
BASE_DIR = cwd
while True:
    if os.path.exists(os.path.join(BASE_DIR, "data", "corpus.json")):
        break
    parent = os.path.dirname(BASE_DIR)
    if parent == BASE_DIR:  # reached root without finding it
        BASE_DIR = cwd  # fallback to cwd
        break
    BASE_DIR = parent
DATA_DIR = os.path.join(BASE_DIR, 'data')
VIZ_DIR = os.path.join(BASE_DIR, 'visualization')
os.makedirs(VIZ_DIR, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"Corpus exists: {os.path.exists(os.path.join(DATA_DIR, 'corpus.json'))}")

BASE_DIR: /content
DATA_DIR: /content/data
Corpus exists: False


## 3. Data Loading & Generation

We'll use a synthetic corpus of data science articles. If it doesn't exist, run `data/generate_data.py` first.

In [37]:
# Walk up from cwd to find project root containing data/corpus.json
BASE_DIR = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(BASE_DIR, "data", "corpus.json")):
        break
    BASE_DIR = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(BASE_DIR, 'data')
VIZ_DIR = os.path.join(BASE_DIR, 'visualization')
os.makedirs(VIZ_DIR, exist_ok=True)
print(f"📂 Data dir: {DATA_DIR}")
print(f"📂 VIZ dir: {VIZ_DIR}")
print("✅ Imports complete")

📂 Data dir: /data
📂 VIZ dir: /visualization
✅ Imports complete


In [51]:
# ===== 3. Data Loading =====

import sys
import json
import random
from pathlib import Path

# Function to generate synthetic corpus
def generate_corpus(n_docs=54, output_path=None):
    """Generate synthetic corpus directly without subprocess."""
    random.seed(42)
    
    documents = {
        "machine_learning": [
            "Supervised learning algorithms learn a mapping from input features to output labels using labeled training data.",
            "Decision trees partition the feature space into regions using a tree-like structure of decisions.",
            "Random forests construct multiple decision trees and output the majority vote of individual trees.",
            "Gradient boosting builds models in a stage-wise fashion where each new model corrects errors.",
            "Support Vector Machines find the optimal hyperplane that maximizes the margin between classes.",
            "Neural networks consist of layers of interconnected neurons that can learn complex patterns.",
            "Overfitting occurs when a model learns training data too well, capturing noise instead of patterns.",
            "Cross-validation is a resampling procedure used to evaluate machine learning models.",
            "Hyperparameter tuning involves finding the optimal set of hyperparameters for an algorithm.",
            "Transfer learning allows pre-trained models to be adapted for new but related tasks.",
        ],
        "deep_learning": [
            "Convolutional Neural Networks use convolutional layers to learn spatial hierarchies from images.",
            "Recurrent Neural Networks are designed to handle sequential data with hidden state.",
            "LSTMs can learn long-term dependencies through gating mechanisms.",
            "Transformers use self-attention mechanisms to process sequences in parallel.",
            "Batch normalization normalizes layer inputs enabling higher learning rates.",
        ],
        "nlp": [
            "Tokenization splits text into smaller units like words, subwords, or characters.",
            "Word embeddings are dense vector representations that capture semantic relationships.",
            "Language models predict the next token given previous tokens in a sequence.",
        ],
        "python": [
            "Python is a versatile programming language popular for data science and web development.",
            "NumPy provides efficient numerical computing with vectorized operations.",
            "Pandas is a powerful library for data manipulation and analysis.",
        ],
    }
    
    corpus = []
    doc_id = 0
    topics = list(documents.keys())
    difficulties = ['beginner', 'intermediate', 'advanced']
    
    for _ in range(n_docs):
        topic = random.choice(topics)
        content = random.choice(documents[topic])
        
        corpus.append({
            'doc_id': f'doc_{doc_id:03d}',
            'title': f'{topic.replace("_", " ").title()} Article {doc_id}',
            'topic': topic,
            'difficulty': random.choice(difficulties),
            'content': content,
            'word_count': len(content.split())
        })
        doc_id += 1
    
    # Generate queries
    queries = []
    for i in range(20):
        topic = random.choice(topics)
        relevant_doc = random.choice([d for d in corpus if d['topic'] == topic])
        queries.append({
            'query_id': f'q_{i:03d}',
            'text': f'Tell me about {topic.replace("_", " ")}',
            'topic': topic,
            'relevant_doc_id': relevant_doc['doc_id']
        })
    
    return corpus, queries

# Generate corpus
print("🔄 Generating synthetic corpus...")
corpus, queries = generate_corpus(n_docs=54)
print(f"✅ Generated {len(corpus)} documents across {len(set(d['topic'] for d in corpus))} topics")
print(f"✅ Generated {len(queries)} evaluation queries")
print(f"📄 Topics: {sorted(set(d['topic'] for d in corpus))}")

🔄 Generating synthetic corpus...
✅ Generated 54 documents across 4 topics
✅ Generated 20 evaluation queries
📄 Topics: ['deep_learning', 'machine_learning', 'nlp', 'python']


## 4. Data Inspection

Let's understand our corpus before building the pipeline.

In [50]:
# ===== 4. Data Inspection =====

df = pd.DataFrame(corpus)
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\n--- First 3 documents ---")
for _, row in df.head(4).iterrows():
    print(f"  [{row['doc_id']}] {row['title']} ({row['topic']}, {row['difficulty']})")
    print(f"      {row['content'][:80]}...")
    print()

Shape: (3, 5)

Columns: ['doc_id', 'title', 'topic', 'difficulty', 'content']

--- First 3 documents ---
  [doc_001] Introduction to Machine Learning (machine_learning, beginner)
      Machine learning is a subset of artificial intelligence that enables systems to ...

  [doc_002] Deep Learning Fundamentals (deep_learning, intermediate)
      Deep learning is a specialized subset of machine learning that uses neural netwo...

  [doc_003] Python for Data Science (python, beginner)
      Python has become the go-to language for data science due to its simplicity and ...



## 5. Exploratory Data Analysis

Understand topic distribution, document lengths, and difficulty levels.

In [ ]:
# ===== 5. EDA =====

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Topic distribution
topic_counts = df['topic'].value_counts()
colors = sns.color_palette('muted', len(topic_counts))
axes[0].barh(topic_counts.index, topic_counts.values, color=colors)
axes[0].set_title('Documents per Topic')
axes[0].set_xlabel('Count')

# Difficulty distribution
diff_counts = df['difficulty'].value_counts()
axes[1].bar(diff_counts.index, diff_counts.values, color=['#10B981', '#F59E0B', '#EF4444'])
axes[1].set_title('Difficulty Distribution')
axes[1].set_ylabel('Count')

# Word count distribution
axes[2].hist(df['word_count'], bins=15, color='#6366F1', edgecolor='white', alpha=0.7)
axes[2].axvline(df['word_count'].mean(), color='#EF4444', linestyle='--', label=f"Mean: {df['word_count'].mean():.0f}")
axes[2].set_title('Document Length (words)')
axes[2].set_xlabel('Word Count')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, '01_corpus_eda.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f"📊 Average document length: {df['word_count'].mean():.0f} words")
print(f"📊 Topic count: {df['topic'].nunique()}")

## 6. Chunking Strategies

Chunking splits documents into smaller pieces that can be retrieved independently. The quality of chunking directly impacts retrieval performance.

**Why chunk?**
- Embedding models have a maximum context length (e.g., 512 tokens)
- Smaller chunks = more precise retrieval (one chunk ≈ one concept)
- Too small chunks lose context; too large chunks dilute relevance

We'll compare **three strategies**:

In [ ]:
# ===== 6. Chunking Strategies =====

def chunk_fixed_size(text: str, chunk_size: int = 100, overlap: int = 20) -> List[str]:
    """Fixed-size character chunking with overlap."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(' '.join(words[start:end]))
        start += chunk_size - overlap
        if start >= len(words):
            break
    return chunks


def chunk_by_sentence(text: str) -> List[str]:
    """Sentence-based chunking using regex split."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sentences if len(s) > 10]


def chunk_recursive(text: str, max_chars: int = 300, overlap_chars: int = 50) -> List[str]:
    """Recursive character splitting with separator awareness."""
    if len(text) <= max_chars:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        if end >= len(text):
            chunks.append(text[start:])
            break
        last_period = text.rfind('.', start, end)
        last_space = text.rfind(' ', start, end)
        split_at = last_period + 1 if last_period > start else last_space
        if split_at <= start:
            split_at = end
        chunks.append(text[start:split_at].strip())
        start = split_at - overlap_chars
    return [c for c in chunks if len(c) > 10]


# Compare strategies on one document
sample_doc = corpus[0]['content']
print(f"📄 Original document ({len(sample_doc.split())} words):")
print(f"   {sample_doc[:120]}...\n")

strategies = {
    'Fixed-size (100 words, 20 overlap)': chunk_fixed_size(sample_doc),
    'Sentence-based': chunk_by_sentence(sample_doc),
    'Recursive character': chunk_recursive(sample_doc),
}

for name, chunks in strategies.items():
    print(f"\n🔹 {name}")
    print(f"   Produced {len(chunks)} chunks")
    for i, c in enumerate(chunks[:3]):
        print(f"   [{i+1}] {c[:70]}...")
    if len(chunks) > 3:
        print(f"   ... and {len(chunks)-3} more")

In [ ]:
# ===== Chunking: Apply to entire corpus =====

def process_corpus(corpus: List[Dict], chunk_fn, **kwargs) -> pd.DataFrame:
    """Apply chunking to all documents and return a DataFrame."""
    rows = []
    for doc in tqdm(corpus, desc="Chunking"):
        chunks = chunk_fn(doc['content'], **kwargs)
        for i, chunk in enumerate(chunks):
            rows.append({
                'chunk_id': f"{doc['doc_id']}_c{i:03d}",
                'doc_id': doc['doc_id'],
                'title': doc['title'],
                'topic': doc['topic'],
                'text': chunk,
                'chunk_index': i,
            })
    return pd.DataFrame(rows)

# Use sentence-based chunking (semantically meaningful boundaries)
chunk_df = process_corpus(corpus, chunk_by_sentence)

print(f"\n✅ Total chunks: {len(chunk_df)}")
print(f"   Average chunk length: {chunk_df['text'].str.split().str.len().mean():.0f} words")
print(f"   Chunks per document: {chunk_df.groupby('doc_id').size().describe()}")

In [ ]:
# ===== Visualize chunk distribution =====

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

chunk_lens = chunk_df['text'].str.split().str.len()
axes[0].hist(chunk_lens, bins=20, color='#8B5CF6', edgecolor='white', alpha=0.7)
axes[0].axvline(chunk_lens.mean(), color='#EF4444', linestyle='--', label=f"Mean: {chunk_lens.mean():.0f}")
axes[0].set_title('Chunk Length Distribution (words)')
axes[0].set_xlabel('Words')
axes[0].set_ylabel('Count')
axes[0].legend()

chunks_per_doc = chunk_df.groupby('doc_id').size()
axes[1].hist(chunks_per_doc, bins=15, color='#F59E0B', edgecolor='white', alpha=0.7)
axes[1].set_title('Chunks per Document')
axes[1].set_xlabel('Number of Chunks')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, '02_chunk_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f"📊 Total chunks: {len(chunk_df)}")
print(f"📊 Mean chunks/doc: {chunks_per_doc.mean():.1f}")

## 7. Embeddings & Vector Store

**Embeddings** convert text chunks into dense numerical vectors that capture semantic meaning. We'll use `all-MiniLM-L6-v2` — a lightweight, efficient sentence-transformer model.

**Vector Store** indexes these embeddings for fast similarity search. We'll use **ChromaDB**, an open-source vector database.

> 💡 **Key insight:** Documents with similar meanings will have similar embedding vectors. Cosine similarity between embeddings measures semantic relatedness.

In [ ]:
# ===== 7. Embeddings & Vector Store =====

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# Load embedding model
print("🔄 Loading embedding model: all-MiniLM-L6-v2...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"✅ Model loaded (embedding dim: {EMBED_DIM})")

In [ ]:
# ===== Create ChromaDB collection =====

# Remove old DB if exists
import shutil
chroma_path = os.path.join(BASE_DIR, 'data', 'chroma_db')
if os.path.exists(chroma_path):
    shutil.rmtree(chroma_path)

# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path=chroma_path, settings=Settings(anonymized_telemetry=False))

# Create collection
collection = chroma_client.create_collection(
    name="rag_corpus",
    metadata={"hnsw:space": "cosine"}
)

print("✅ ChromaDB collection created")

In [ ]:
# ===== Compute embeddings and populate vector store =====

texts = chunk_df['text'].tolist()
ids = chunk_df['chunk_id'].tolist()
metadatas = chunk_df[['doc_id', 'title', 'topic', 'chunk_index']].to_dict('records')

print(f"🔄 Embedding {len(texts)} chunks...")
batch_size = 32
for i in tqdm(range(0, len(texts), batch_size), desc="Embedding & Indexing"):
    batch_texts = texts[i:i+batch_size]
    batch_ids = ids[i:i+batch_size]
    batch_metadatas = metadatas[i:i+batch_size]
    embeddings = embed_model.encode(batch_texts).tolist()
    collection.add(
        embeddings=embeddings,
        documents=batch_texts,
        ids=batch_ids,
        metadatas=batch_metadatas
    )

print(f"✅ Indexed {collection.count()} chunks in ChromaDB")

## 8. Retrieval

Now we retrieve the most relevant chunks for a query. We'll test two approaches:

1. **Dense retrieval** — cosine similarity on dense embeddings (semantic search)
2. **Hybrid retrieval** — combining dense + keyword (BM25) scores

Let's first test retrieval on a few example queries.

In [ ]:
# ===== 8. Retrieval =====

def retrieve(query: str, k: int = 5) -> pd.DataFrame:
    """Retrieve top-k chunks for a query using ChromaDB."""
    query_emb = embed_model.encode([query])[0].tolist()
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=k,
        include=['documents', 'metadatas', 'distances']
    )
    rows = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        rows.append({
            'chunk_id': meta.get('chunk_id', ''),
            'title': meta.get('title', ''),
            'topic': meta.get('topic', ''),
            'text': doc[:150] + '...',
            'score': 1 - dist,  # Convert distance to similarity
        })
    return pd.DataFrame(rows)


def display_retrieval(query: str, results: pd.DataFrame):
    """Pretty-print retrieval results."""
    print(f"🔍 Query: \"{query}\"\n")
    for i, (_, row) in enumerate(results.iterrows()):
        print(f"  #{i+1} [score={row['score']:.4f}] ({row['topic']})")
        print(f"      {row['title']}")
        print(f"      {row['text']}\n")


# Test queries
test_queries = [
    "What is the difference between random forests and gradient boosting?",
    "How does the attention mechanism work in transformers?",
    "What is the purpose of batch normalization in neural networks?",
    "How do you handle overfitting in machine learning models?",
]

for q in test_queries:
    results = retrieve(q, k=3)
    display_retrieval(q, results)
    print("---" * 20)

### 8.1 Hybrid Retrieval (Dense + Keyword)

Pure dense retrieval can miss exact keyword matches. Hybrid retrieval combines:
- **Dense score** — semantic similarity (from our embeddings)
- **Keyword score** — lexical overlap (BM25-style term matching)

This gives us the best of both worlds.

In [ ]:
# ===== 8.1 Hybrid Retrieval =====

from collections import Counter
import math


def compute_bm25_scores(query: str, documents: List[str], k1: float = 1.5, b: float = 0.75) -> np.ndarray:
    """Simplified BM25 scoring for a single query against a list of documents."""
    query_terms = query.lower().split()
    n_docs = len(documents)
    avg_dl = np.mean([len(d.split()) for d in documents])
    
    # IDF for each query term
    doc_term_counts = []
    for doc in documents:
        doc_term_counts.append(Counter(doc.lower().split()))
    
    scores = np.zeros(n_docs)
    for term in query_terms:
        df = sum(1 for dtc in doc_term_counts if term in dtc)
        idf = math.log((n_docs - df + 0.5) / (df + 0.5) + 1.0)
        for i, dtc in enumerate(doc_term_counts):
            tf = dtc.get(term, 0)
            if tf > 0:
                dl = sum(dtc.values())
                scores[i] += idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avg_dl))
    return scores


def hybrid_retrieve(query: str, k: int = 5, alpha: float = 0.7) -> pd.DataFrame:
    """Hybrid retrieval combining dense (alpha) and keyword (1-alpha) scores."""
    # Dense retrieval: get more candidates for reranking
    query_emb = embed_model.encode([query])[0].tolist()
    dense_results = collection.query(
        query_embeddings=[query_emb],
        n_results=20,
        include=['documents', 'metadatas', 'distances']
    )
    
    dense_docs = dense_results['documents'][0]
    dense_metas = dense_results['metadatas'][0]
    dense_scores = 1 - np.array(dense_results['distances'][0])
    
    # BM25 scores
    keyword_scores = compute_bm25_scores(query, dense_docs)
    
    # Normalize scores to [0, 1]
    def normalize(scores):
        mn, mx = scores.min(), scores.max()
        if mx == mn:
            return np.ones_like(scores)
        return (scores - mn) / (mx - mn)
    
    dense_norm = normalize(dense_scores)
    keyword_norm = normalize(keyword_scores)
    
    # Combine
    combined = alpha * dense_norm + (1 - alpha) * keyword_norm
    top_indices = np.argsort(combined)[::-1][:k]
    
    rows = []
    for idx in top_indices:
        rows.append({
            'chunk_id': dense_metas[idx].get('chunk_id', ''),
            'title': dense_metas[idx].get('title', ''),
            'topic': dense_metas[idx].get('topic', ''),
            'text': dense_docs[idx][:150] + '...',
            'dense_score': dense_norm[idx],
            'keyword_score': keyword_norm[idx],
            'combined_score': combined[idx],
        })
    return pd.DataFrame(rows)


# Compare dense vs hybrid on a test query
test_q = "How does gradient boosting differ from random forest?"

print("🔍", test_q, "\n")

print("--- Dense Retrieval ---")
dense_res = retrieve(test_q, k=3)
display_retrieval(test_q, dense_res)

print("\n--- Hybrid Retrieval (alpha=0.7) ---")
hybrid_res = hybrid_retrieve(test_q, k=3)
for i, (_, row) in enumerate(hybrid_res.iterrows()):
    print(f"  #{i+1} [combined={row['combined_score']:.4f}] ({row['topic']})")
    print(f"      dense={row['dense_score']:.3f}, keyword={row['keyword_score']:.3f}")
    print(f"      {row['title']}")
    print(f"      {row['text']}\n")

## 9. Reranking

Retrieval gives us candidates, but ordering by embedding similarity isn't always optimal for the final task. **Reranking** applies a more expensive, more accurate model (cross-encoder) to re-score the top candidates.

**Cross-encoders** take a query-document pair as input and output a relevance score. They're slower but more accurate than bi-encoders (our embedding model).

In [ ]:
# ===== 9. Reranking =====

from sentence_transformers import CrossEncoder

print("🔄 Loading cross-encoder reranker...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Reranker loaded")

In [ ]:
# ===== Reranking in action =====

def retrieve_and_rerank(query: str, k_retrieve: int = 10, k_final: int = 3) -> pd.DataFrame:
    """Retrieve candidates, then rerank with cross-encoder."""
    # First pass: dense retrieval
    query_emb = embed_model.encode([query])[0].tolist()
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=k_retrieve,
        include=['documents', 'metadatas', 'distances']
    )
    
    docs = results['documents'][0]
    metas = results['metadatas'][0]
    dense_scores = 1 - np.array(results['distances'][0])
    
    # Rerank: cross-encoder scores
    pairs = [[query, doc] for doc in docs]
    rerank_scores = reranker.predict(pairs)
    
    top_indices = np.argsort(rerank_scores)[::-1][:k_final]
    
    rows = []
    for idx in top_indices:
        rows.append({
            'chunk_id': metas[idx].get('chunk_id', ''),
            'title': metas[idx].get('title', ''),
            'topic': metas[idx].get('topic', ''),
            'text': docs[idx][:150] + '...',
            'dense_score': dense_scores[idx],
            'rerank_score': rerank_scores[idx],
        })
    return pd.DataFrame(rows)


# Compare before and after reranking
query = "What is the attention mechanism and how does it work?"

print("🔍", query, "\n")

# Before reranking (dense only)
print("--- Before Reranking (Dense Scores) ---")
dense = retrieve(query, k=5)
for i, (_, row) in enumerate(dense.iterrows()):
    print(f"  #{i+1} [score={row['score']:.4f}] {row['title'][:60]}")

# After reranking
print("\n--- After Reranking ---")
reranked = retrieve_and_rerank(query, k_retrieve=10, k_final=5)
for i, (_, row) in enumerate(reranked.iterrows()):
    print(f"  #{i+1} [rerank={row['rerank_score']:.4f}, dense={row['dense_score']:.4f}] {row['title'][:60]}")

## 10. Generation (Augmented)

Now we augment the retrieved context and feed it to a language model for answer generation. We'll use a small T5 model that runs on CPU — demonstrating the full RAG cycle.

**The RAG prompt template:**
```
Context:
[retrieved chunk 1]
[retrieved chunk 2]
...

Question: [user query]

Answer the question based only on the provided context.
```

In [ ]:
# ===== 10. Generation =====

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Use a small T5 model that can run on CPU
MODEL_NAME = 'google/flan-t5-small'

print(f"🔄 Loading generation model: {MODEL_NAME}...")
gen_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print("✅ Generator loaded")

In [ ]:
# ===== RAG Generation in action =====

def rag_answer(query: str, k: int = 3, max_context: int = 800) -> str:
    """Full RAG pipeline: retrieve → augment → generate."""
    # Step 1: Retrieve
    results = retrieve(query, k=k)
    
    # Step 2: Augment — build context from retrieved chunks
    context_parts = []
    total_chars = 0
    for _, row in results.iterrows():
        chunk_text = row['text'].replace('...', '')
        if total_chars + len(chunk_text) > max_context:
            break
        context_parts.append(chunk_text)
        total_chars += len(chunk_text)
    context = ' '.join(context_parts)
    
    # Step 3: Generate
    prompt = f"""Answer the question based on the provided context.

Context: {context}

Question: {query}

Answer:"""
    
    inputs = gen_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024)
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.3,
        do_sample=True,
    )
    answer = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer, results


# Test with a few queries
test_queries = [
    "What is overfitting in machine learning and how can we prevent it?",
    "How does transfer learning help in deep learning?",
]

for q in test_queries:
    print(f"\n{'='*60}")
    print(f"🔍 Query: {q}")
    answer, retrieved = rag_answer(q)
    print(f"\n📖 Retrieved context from:")
    for _, row in retrieved.iterrows():
        print(f"   - {row['title'][:60]} (score={row['score']:.3f})")
    print(f"\n🤖 Generated answer: {answer}")

### 10.1 Compare: With vs Without RAG

Let's see what happens when the model answers **without** retrieval context. This demonstrates why RAG matters — the model must rely only on its limited parametric knowledge.

In [ ]:
# ===== 10.1 Without RAG (parametric only) =====

def answer_no_context(query: str) -> str:
    """Generate answer without retrieval — relying only on model parameters."""
    prompt = f"Answer the question: {query}"
    inputs = gen_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.3,
        do_sample=True,
    )
    return gen_tokenizer.decode(outputs[0], skip_special_tokens=True)


queries_to_compare = [
    "What is the difference between bagging and boosting?",
    "Explain how LSTMs handle the vanishing gradient problem.",
]

for q in queries_to_compare:
    print(f"\n{'='*60}")
    print(f"🔍 Query: {q}")
    
    print("\n🤖 Without RAG (parametric only):")
    no_rag = answer_no_context(q)
    print(f"   {no_rag}")
    
    print("\n📚 With RAG (retrieved + generated):")
    rag_ans, _ = rag_answer(q)
    print(f"   {rag_ans}")

## 11. Evaluation

We need to measure how well our RAG pipeline performs. Two key dimensions:

1. **Retrieval quality** — Are we getting the right documents?
2. **Generation quality** — Are the answers accurate?

For retrieval, we'll compute **Precision@k** and **Recall@k** using our labeled query-document pairs.

In [ ]:
# ===== 11. Evaluation =====

def evaluate_retrieval(queries: List[Dict], k_values: List[int] = [1, 3, 5]) -> pd.DataFrame:
    """Evaluate retrieval precision and recall at different k values."""
    results = []
    
    for q in tqdm(queries, desc="Evaluating retrieval"):
        query_text = q['text']
        relevant_doc_id = q['relevant_doc_id']
        
        # Retrieve
        query_emb = embed_model.encode([query_text])[0].tolist()
        retrieved = collection.query(
            query_embeddings=[query_emb],
            n_results=max(k_values),
            include=['metadatas']
        )
        
        retrieved_doc_ids = [m.get('doc_id', '') for m in retrieved['metadatas'][0]]
        
        for k in k_values:
            retrieved_k = retrieved_doc_ids[:k]
            relevant_count = sum(1 for doc_id in retrieved_k if doc_id == relevant_doc_id)
            precision = relevant_count / k
            recall = 1.0 if relevant_count > 0 else 0.0
            
            results.append({
                'query_id': q['query_id'],
                'topic': q['topic'],
                'k': k,
                'precision': precision,
                'recall': recall,
                'hit': int(relevant_count > 0),
            })
    
    return pd.DataFrame(results)


eval_df = evaluate_retrieval(queries, k_values=[1, 3, 5])

# Summary stats
summary = eval_df.groupby('k').agg({
    'precision': 'mean',
    'recall': 'mean',
    'hit': 'mean'
}).round(4)
summary.columns = ['Precision@k', 'Recall@k', 'HitRate@k']
print("\n=== Retrieval Evaluation Results ===")
print(summary.to_string())

In [ ]:
# ===== Visualize evaluation results =====

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision/Recall by k
plot_df = eval_df.groupby('k')[['precision', 'recall']].mean().reset_index()
axes[0].plot(plot_df['k'], plot_df['precision'], 'o-', color='#6366F1', linewidth=2, label='Precision@k')
axes[0].plot(plot_df['k'], plot_df['recall'], 's-', color='#10B981', linewidth=2, label='Recall@k')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Score')
axes[0].set_title('Retrieval Performance vs k')
axes[0].set_xticks([1, 3, 5])
axes[0].legend()
axes[0].set_ylim(0, 1)

# Hit rate by topic
topic_hit = eval_df[eval_df['k'] == 3].groupby('topic')['hit'].mean().sort_values()
axes[1].barh(topic_hit.index, topic_hit.values, color=sns.color_palette('muted', len(topic_hit)))
axes[1].set_title('HitRate@3 by Topic')
axes[1].set_xlabel('Hit Rate')
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, '03_retrieval_evaluation.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f"📊 Average Precision@3: {eval_df[eval_df['k']==3]['precision'].mean():.3f}")
print(f"📊 Average Recall@3: {eval_df[eval_df['k']==3]['recall'].mean():.3f}")

### 11.1 Ablation: Compare Chunking Strategies

Let's evaluate how different chunking strategies affect retrieval performance.

In [ ]:
# ===== 11.1 Chunking strategy comparison =====

def evaluate_chunking_strategy(corpus, chunk_fn, strategy_name, k=3, **kwargs):
    """Evaluate a chunking strategy by rebuilding the index and measuring retrieval."""
    print(f"🔬 Evaluating: {strategy_name}")
    
    # Chunk
    chunk_df = process_corpus(corpus, chunk_fn, **kwargs)
    
    # Create temp collection
    temp_name = f"eval_{strategy_name.replace(' ', '_')}"
    try:
        chroma_client.delete_collection(temp_name)
    except:
        pass
    temp_col = chroma_client.create_collection(name=temp_name, metadata={"hnsw:space": "cosine"})
    
    # Embed and index
    texts = chunk_df['text'].tolist()
    ids = chunk_df['chunk_id'].tolist()
    metas = chunk_df[['doc_id', 'title', 'topic']].to_dict('records')
    
    for i in range(0, len(texts), 32):
        batch_emb = embed_model.encode(texts[i:i+32]).tolist()
        temp_col.add(
            embeddings=batch_emb,
            documents=texts[i:i+32],
            ids=ids[i:i+32],
            metadatas=metas[i:i+32]
        )
    
    # Evaluate
    hits = []
    for q in queries:
        q_emb = embed_model.encode([q['text']])[0].tolist()
        res = temp_col.query(query_embeddings=[q_emb], n_results=k, include=['metadatas'])
        retrieved_ids = [m.get('doc_id', '') for m in res['metadatas'][0]]
        hits.append(1 if q['relevant_doc_id'] in retrieved_ids else 0)
    
    chroma_client.delete_collection(temp_name)
    return np.mean(hits)


# Compare three chunking strategies
strategies_to_test = [
    ('Fixed-size (100 words)', chunk_fixed_size, {'chunk_size': 100, 'overlap': 20}),
    ('Sentence-based', chunk_by_sentence, {}),
    ('Recursive character', chunk_recursive, {'max_chars': 300, 'overlap_chars': 50}),
]

print("Comparing chunking strategies (may take a minute)...\n")
for name, fn, kwargs in strategies_to_test:
    hit_rate = evaluate_chunking_strategy(corpus, fn, name, k=3, **kwargs)
    print(f"   HitRate@3 for '{name}': {hit_rate:.3f}")
    print()

## 12. Conclusion & Insights

We've built a complete **RAG pipeline from scratch** — from corpus to chunking to embedding to retrieval to generation to evaluation.

### Key Takeaways

| Component | Key Insight |
|:----------|:------------|
| **Chunking** | Sentence-based chunking gives semantically coherent units; fixed-size is simpler but can cut through meaning |
| **Embeddings** | Dense embeddings (all-MiniLM-L6-v2) capture semantic similarity beyond keyword overlap |
| **Vector Store** | ChromaDB provides fast approximate nearest neighbor search with cosine similarity |
| **Retrieval** | Dense retrieval works for semantic search; hybrid (dense + BM25) improves recall for exact terms |
| **Reranking** | Cross-encoders significantly improve top-k relevance at the cost of additional computation |
| **Generation** | RAG-grounded answers are more factual than parametric-only generation, especially for niche topics |

### Production Considerations

| Concern | Recommendation |
|:--------|:---------------|
| Scale | Use a production vector DB (Qdrant, Pinecone, Weaviate) with GPU-accelerated indexing |
| Accuracy | Fine-tune the embedding model on your domain; use a stronger reranker |
| Latency | Cache embeddings; use approximate nearest neighbor (HNSW/IVF) indexes |
| Data freshness | Implement incremental indexing for new documents without full rebuild |
| LLM quality | Replace flan-t5-small with GPT-4, Claude, or fine-tuned open-source models |

### What's Next?

- ✅ Add **query expansion** (generate multiple query variants for better recall)
- ✅ Implement **HyDE** (Hypothetical Document Embeddings) retrieval
- ✅ Add **self-querying retrieval** (extract filters from natural language queries)
- ✅ Build a **gradio/streamlit UI** for interactive Q&A
- ✅ Add **agentic RAG** where the LLM decides when to retrieve

In [ ]:
# ===== 12. Summary =====

print("=" * 60)
print("📋 RAG PIPELINE — PROJECT SUMMARY")
print("=" * 60)
print(f"\n📄 Corpus: {len(corpus)} documents across {df['topic'].nunique()} topics")
print(f"🔪 Chunks: {len(chunk_df)} chunks ({chunk_df['text'].str.split().str.len().mean():.0f} avg words)")
print(f"🧠 Embedding model: all-MiniLM-L6-v2 (dim={EMBED_DIM})")
print(f"🗄️  Vector store: ChromaDB (cosine similarity, {collection.count()} vectors)")
print(f"⚡ Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2")
print(f"🤖 Generator: {MODEL_NAME}")
print(f"\n📊 Retrieval Performance:")

for k in [1, 3, 5]:
    sub = eval_df[eval_df['k'] == k]
    print(f"   Precision@{k}: {sub['precision'].mean():.3f}  |  Recall@{k}: {sub['recall'].mean():.3f}")

print(f"\n✅ RAG pipeline complete! All components working end-to-end.")